# HCR register — exploration scratchpad

Exploration only; **nothing load-bearing lives here**. Every result that
matters is computed by the `hcr` package and written up in `reports/`.
This notebook exists to poke at the cleaned data interactively.

In [ ]:
import pandas as pd

from hcr import ingest, clean, profile, validate

ingest.inventory()

In [ ]:
df = pd.concat(
    [clean.clean_records(t["frame"], year=t["years"][0])
     for t in ingest.load_source_tables()],
    ignore_index=True,
)
dfy = profile.with_event_year(df)
dfy.shape

In [ ]:
validate.run_all(dfy)

In [ ]:
# the hole-size round-down step
profile.boundary_concentration(dfy, "hole_diameter_mm", 1.0, 2.0)

In [ ]:
# exact-value distribution: look for the imperial anchors (6.35, 12.7, 25.4, 50.8)
nd = profile.numeric_distribution(dfy, "hole_diameter_mm")
nd.sort_values("count", ascending=False).head(15)

In [ ]:
# missingness by year — watch quantity_released_kg collapse after 2018
profile.missingness_by_column_by_year(
    dfy[["hole_diameter_mm", "quantity_released_kg", "severity", "year"]], "year"
).round(3)

In [ ]:
# severity drift across the 1999 criteria change
profile.category_drift(dfy, "severity", "year").round(3)